In [ ]:
import numpy as np
import pandas as pd

# ==============================================================================
# 1. HÀM CHUẨN HÓA VÀ TÍNH KPI CHI TIẾT CHO SHIPPER (Statement 1)
# ==============================================================================
def calculate_shipper_kpis(df):
    """
    Tính toàn bộ KPI cho từng Shipper, bao gồm Median và Percentile 90
    df yêu cầu các cột: [ShipperID, OrderID, CustomerID, OrderValue,
                         DeliveryTime_Hours, IsSuccess, IsOnTime, IsLate,
                         IsReturn, IsComplaint, Rating]
    """
    total_sys_rev = df[df['IsSuccess'] == 1]['OrderValue'].sum()

    shipper_kpi = df.groupby('ShipperID').agg(
        Total_Orders=('OrderID', 'count'),
        Success_Orders=('IsSuccess', 'sum'),
        Total_Revenue=('OrderValue', lambda x: x[df.loc[x.index, 'IsSuccess'] == 1].sum()),
        Customers_Served=('CustomerID', 'nunique'),
        OnTime_Orders=('IsOnTime', 'sum'),
        Late_Orders=('IsLate', 'sum'),
        Return_Orders=('IsReturn', 'sum'),
        Complaint_Orders=('IsComplaint', 'sum'),
        Avg_Rating=('Rating', 'mean'),
        Avg_Delivery_Time=('DeliveryTime_Hours', 'mean'),
        Median_Delivery_Time=('DeliveryTime_Hours', 'median'),
        P90_Delivery_Time=('DeliveryTime_Hours', lambda x: np.percentile(x.dropna(), 90))
    ).reset_index()

    # Tính các tỷ lệ phần trăm (0 - 100%)
    shipper_kpi['Success_Rate'] = (shipper_kpi['Success_Orders'] / shipper_kpi['Total_Orders']) * 100
    shipper_kpi['OnTime_Rate'] = (shipper_kpi['OnTime_Orders'] / shipper_kpi['Total_Orders']) * 100
    shipper_kpi['Late_Rate'] = (shipper_kpi['Late_Orders'] / shipper_kpi['Total_Orders']) * 100
    shipper_kpi['Return_Rate'] = (shipper_kpi['Return_Orders'] / shipper_kpi['Total_Orders']) * 100
    shipper_kpi['Complaint_Rate'] = (shipper_kpi['Complaint_Orders'] / shipper_kpi['Total_Orders']) * 100
    shipper_kpi['Revenue_Contribution'] = (shipper_kpi['Total_Revenue'] / total_sys_rev) * 100

    return shipper_kpi


# ==============================================================================
# 2. TÍNH KPI SCORE & PHÂN LOẠI SHIPPER (Statement 2)
# ==============================================================================
def score_and_classify_shippers(df_kpi):
    """
    Tính điểm KPI Score có đảo chiều chỉ số tiêu cực và phân nhóm hiệu suất
    """
    kpi = df_kpi.copy()

    def min_max_norm(series, invert=False):
        min_val, max_val = series.min(), series.max()
        if max_val == min_val:
            return pd.Series(100.0, index=series.index)
        if invert:
            return ((max_val - series) / (max_val - min_val)) * 100
        return ((series - min_val) / (max_val - min_val)) * 100

    # Chuẩn hóa các chỉ số về thang điểm 0 - 100
    norm_success = min_max_norm(kpi['Success_Rate'])
    norm_ontime = min_max_norm(kpi['OnTime_Rate'])
    norm_rating = min_max_norm(kpi['Avg_Rating'])
    norm_revenue = min_max_norm(kpi['Revenue_Contribution'])

    # Chỉ số tiêu cực cần đảo chiều: Thời gian giao hàng (Median) và Tỷ lệ khiếu nại
    norm_speed = min_max_norm(kpi['Median_Delivery_Time'], invert=True)
    norm_complaint = min_max_norm(kpi['Complaint_Rate'], invert=True)

    # Công thức trọng số theo hướng dẫn
    kpi['KPI_Score'] = (
        0.25 * norm_success +
        0.20 * norm_ontime +
        0.20 * norm_rating +
        0.15 * norm_speed +
        0.10 * norm_complaint +
        0.10 * norm_revenue
    )

    # Phân loại Rule-based / Percentile
    def classify(score):
        if score >= 80:
            return 'Top Performers'
        elif score >= 65:
            return 'High Performers'
        elif score >= 50:
            return 'Average'
        else:
            return 'Underperformers'

    kpi['Performance_Tier'] = kpi['KPI_Score'].apply(classify)
    return kpi.sort_values(by='KPI_Score', ascending=False).reset_index(drop=True)


# ==============================================================================
# 3. THEO DÕI XU HƯỚNG THEO THỜI GIAN & CẢNH BÁO (Statement 3)
# ==============================================================================
def track_trends_and_alerts(df):
    """
    Tính rolling average và phát hiện shipper giảm hiệu suất liên tiếp
    """
    df['YearMonth'] = pd.to_datetime(df['OrderDate']).dt.to_period('M')

    monthly = df.groupby(['ShipperID', 'YearMonth']).agg(
        Orders=('OrderID', 'count'),
        Success_Rate=('IsSuccess', lambda x: x.mean() * 100),
        Avg_Rating=('Rating', 'mean'),
        Median_Time=('DeliveryTime_Hours', 'median')
    ).reset_index()

    monthly['Success_Rate_MA3'] = monthly.groupby('ShipperID')['Success_Rate'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )

    # Kiểm tra suy giảm liên tiếp: kỳ này thấp hơn kỳ trước
    monthly['Success_Drop'] = monthly.groupby('ShipperID')['Success_Rate'].diff() < 0

    return monthly


# ==============================================================================
# 4. PHÂN TÍCH PHẢN HỒI (SENTIMENT & COMPLAINT CATEGORIES) (Statement 4)
# ==============================================================================
def analyze_customer_feedback(df_comments):
    """
    Phân loại nguyên nhân khiếu nại dựa trên từ khóa tiếng Việt chuẩn hóa
    """
    categories = {
        'Giao chậm': ['chậm', 'lâu', 'muộn', 'trễ', 'chờ mòn mỏi'],
        'Thái độ': ['thái độ', 'bất lịch sự', 'cộc cằn', 'quát', 'khó chịu', 'chửi'],
        'Hư hỏng': ['bể', 'vỡ', 'móp', 'rách', 'hư hỏng', 'dập'],
        'Sai hàng': ['sai hàng', 'nhầm hàng', 'thiếu hàng'],
        'Liên lạc khó': ['không gọi', 'không liên lạc', 'tự ý hủy', 'không nghe máy']
    }

    def categorize_comment(text):
        if not isinstance(text, str):
            return 'Khác / Không rõ'
        text_lower = text.lower()
        matched = []
        for cat, kw_list in categories.items():
            if any(kw in text_lower for kw in kw_list):
                matched.append(cat)
        return ', '.join(matched) if matched else 'Phản hồi bình thường'

    res = df_comments.copy()
    res['Complaint_Category'] = res['Comment'].apply(categorize_comment)

    # Báo cáo tỷ lệ nguyên nhân theo đối tác vận chuyển / shipper
    complaint_summary = res[res['Complaint_Category'] != 'Phản hồi bình thường'].groupby(
        ['CarrierName', 'Complaint_Category']
    )['OrderID'].count().reset_index(name='Issue_Count')

    return complaint_summary


# ==============================================================================
# 5. SO SÁNH CÔNG TY VẬN CHUYỂN CHUẨN HÓA THEO ĐỘ KHÓ (Statement 5)
# ==============================================================================
def compare_carriers_adjusted(df):
    """
    So sánh công ty vận chuyển có chuẩn hóa độ khó (Ngoại thành, Khoảng cách)
    """
    carrier_kpi = df.groupby(['CarrierName', 'AreaType']).agg(
        Total_Orders=('OrderID', 'count'),
        Success_Rate=('IsSuccess', lambda x: x.mean() * 100),
        Avg_Time=('DeliveryTime_Hours', 'mean'),
        Median_Time=('DeliveryTime_Hours', 'median'),
        Complaint_Rate=('IsComplaint', lambda x: x.mean() * 100),
        Rating=('Rating', 'mean')
    ).reset_index()

    return carrier_kpi.sort_values(by=['AreaType', 'Success_Rate'], ascending=[True, False])


# ==============================================================================
# CHẠY THỬ NGHIỆM MẪU DỮ LIỆU TỔNG HỢP
# ==============================================================================
if __name__ == '__main__':
    np.random.seed(42)
    n = 500
    mock_data = pd.DataFrame({
        'OrderID': range(1, n + 1),
        'ShipperID': np.random.choice([f'SHIP_{i:02d}' for i in range(1, 11)], n),
        'CarrierName': np.random.choice(['GHTK', 'GHN', 'Viettel Post', 'J&T'], n),
        'CustomerID': np.random.randint(100, 200, n),
        'OrderDate': pd.date_range('2026-01-01', periods=n, freq='4h'),
        'OrderValue': np.random.uniform(100000, 2000000, n),
        'AreaType': np.random.choice(['Nội thành', 'Ngoại thành/Vùng xa'], n, p=[0.65, 0.35]),
        'DeliveryTime_Hours': np.random.exponential(scale=12, size=n) + 2,
        'IsSuccess': np.random.choice([1, 0], n, p=[0.92, 0.08]),
        'IsOnTime': np.random.choice([1, 0], n, p=[0.88, 0.12]),
        'IsLate': np.random.choice([1, 0], n, p=[0.10, 0.90]),
        'IsReturn': np.random.choice([1, 0], n, p=[0.05, 0.95]),
        'IsComplaint': np.random.choice([1, 0], n, p=[0.07, 0.93]),
        'Rating': np.random.choice([1, 2, 3, 4, 5], n, p=[0.03, 0.05, 0.12, 0.30, 0.50]),
        'Comment': np.random.choice([
            'Giao hàng nhanh shipper nhiệt tình',
            'Giao quá trễ hàng bị móp méo hộp',
            'Thái độ shipper cộc cằn quát khách',
            'Không gọi điện trước tự ý báo hủy',
            'Hàng nguyên vẹn đóng gói cẩn thận'
        ], n)
    })

    # 1. Tính KPI & Phân loại Shipper
    kpis = calculate_shipper_kpis(mock_data)
    ranked_shippers = score_and_classify_shippers(kpis)
    print("=== TOP 5 SHIPPER THEO KPI SCORE ===")
    print(ranked_shippers[['ShipperID', 'KPI_Score', 'Performance_Tier', 'Success_Rate', 'Median_Delivery_Time', 'Complaint_Rate']].head())

    # 2. So sánh công ty theo khu vực (độ khó)
    carrier_comp = compare_carriers_adjusted(mock_data)
    print("\n=== SO SÁNH CÔNG TY GIAO HÀNG THEO KHU VỰC ===")
    print(carrier_comp.head(6))